# VQ-VAE Interpretability Analysis

Understand what the 8-token bottleneck learns:
1. **Slot-Codebook Usage Matrix** (8 x 256) — does each slot specialize on a subset of codes?
2. **Per-slot entropy** — how diverse is each slot's codebook usage?
3. **Co-occurrence matrix** (K x K) — which codes appear together in the same frame?
4. **Dead codes** — which codebook entries are never used?
5. **Token-to-pitch mapping** — what does each code "look like" on the pitch?

In [ ]:
# -- Setup (run once) --
!rm -rf thesis/
!git clone https://github.com/fegerar/thesis
!cd thesis && git checkout main

In [ ]:
!pip install -r thesis/requirements.txt
!pip install gdown

In [ ]:
# Download dataset
!gdown 1qnG7H2Ro1DdRBcBW7AYwHj9LPhA73uir

In [ ]:
import sys, os
os.chdir("thesis")
sys.path.insert(0, "src")

import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_batch

from vqvae import VQVAELightningModule
from vqvae.dataset import load_shapegraphs, ShapegraphDataset, PITCH_X, PITCH_Y

In [ ]:
# ---- CONFIG: set these ----
CONFIG_PATH = "config/vqvae_pos_only.yml"
CHECKPOINT_PATH = "/content/drive/MyDrive/thesis/checkpoints/vqvae/best.ckpt"  # adjust
DATA_PATH = "/content/shapegraphs.pkl"
SPLIT = "val"  # which split to analyze
MAX_BATCHES = None  # set to e.g. 200 to speed up; None = full split

In [ ]:
# Load config, data, and model
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

data_cfg = cfg["data"]
all_data, node_dim = load_shapegraphs(DATA_PATH)
n = len(all_data)

generator = torch.Generator().manual_seed(data_cfg["seed"])
indices = torch.randperm(n, generator=generator).tolist()
n_train = int(n * data_cfg["train_ratio"])
n_val = int(n * data_cfg["val_ratio"])

splits = {
    "train": indices[:n_train],
    "val": indices[n_train:n_train + n_val],
    "test": indices[n_train + n_val:],
}
split_data = [all_data[i] for i in splits[SPLIT]]
print(f"Split '{SPLIT}': {len(split_data)} frames")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lit_model = VQVAELightningModule.load_from_checkpoint(CHECKPOINT_PATH, map_location=device)
lit_model.eval()
lit_model.to(device)
model = lit_model.model

K = model.quantizer.K
T = model.encoder.num_summary_tokens
print(f"Codebook size K={K}, tokens per frame T={T}")

## Collect tokens and ground-truth features for all frames

In [ ]:
loader = DataLoader(ShapegraphDataset(split_data), batch_size=256, shuffle=False, num_workers=2)

all_tokens = []   # list of (B, T) tensors
all_gt = []       # list of (B, N_max, 4) dense ground-truth

with torch.no_grad():
    for i, batch in enumerate(loader):
        if MAX_BATCHES is not None and i >= MAX_BATCHES:
            break
        batch = batch.to(device)
        tokens = model.encode(batch.x, batch.edge_index, batch.batch)  # (B, T)
        all_tokens.append(tokens.cpu())

        x_dense, mask = to_dense_batch(batch.x, batch.batch)  # (B, N_max, 4)
        all_gt.append(x_dense.cpu())

all_tokens = torch.cat(all_tokens, dim=0)  # (N_frames, T)
all_gt = torch.cat(all_gt, dim=0)          # (N_frames, N_max, 4)
print(f"Collected tokens: {all_tokens.shape}, GT: {all_gt.shape}")

## 1. Slot-Codebook Usage Matrix

Each row = one of the 8 token slots. Each column = one of the K codebook entries.
Cell value = how many frames assigned that code to that slot.

**What to look for:**
- Bright horizontal bands = slot specialization (good — each slot "means" something different)
- Identical rows = redundant slots (bad — wasted capacity)
- Many dark columns = dead codes (unused codebook entries)

In [ ]:
# Build slot-codebook usage matrix (T x K)
usage_matrix = np.zeros((T, K), dtype=np.int64)
for slot in range(T):
    codes = all_tokens[:, slot].numpy()
    for code in codes:
        usage_matrix[slot, code] += 1

# Sort columns by total usage for cleaner visualization
col_order = np.argsort(-usage_matrix.sum(axis=0))
usage_sorted = usage_matrix[:, col_order]

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Raw matrix
im0 = axes[0].imshow(usage_matrix, aspect="auto", cmap="viridis", interpolation="nearest")
axes[0].set_xlabel("Codebook index")
axes[0].set_ylabel("Token slot")
axes[0].set_yticks(range(T))
axes[0].set_title("Slot-Codebook Usage (raw order)")
plt.colorbar(im0, ax=axes[0], label="Count")

# Sorted by total usage
im1 = axes[1].imshow(usage_sorted, aspect="auto", cmap="viridis", interpolation="nearest")
axes[1].set_xlabel("Codebook index (sorted by total usage)")
axes[1].set_ylabel("Token slot")
axes[1].set_yticks(range(T))
axes[1].set_title("Slot-Codebook Usage (sorted)")
plt.colorbar(im1, ax=axes[1], label="Count")

plt.tight_layout()
plt.show()

# Print top-5 codes per slot
for slot in range(T):
    top5 = np.argsort(-usage_matrix[slot])[:5]
    top5_counts = usage_matrix[slot, top5]
    total = usage_matrix[slot].sum()
    top5_pct = top5_counts.sum() / total * 100
    print(f"Slot {slot}: top-5 codes {list(top5)} cover {top5_pct:.1f}% of frames")

## 2. Per-Slot Entropy

Entropy measures how "spread out" each slot's codebook usage is.
- Max entropy = log2(K) = 8.0 bits (uniform over 256 codes)
- Low entropy = slot collapsed to a few codes (specialized or degenerate)

In [ ]:
from scipy.stats import entropy as sp_entropy

slot_entropies = []
for slot in range(T):
    counts = usage_matrix[slot].astype(np.float64)
    probs = counts / counts.sum()
    ent = sp_entropy(probs, base=2)
    slot_entropies.append(ent)

max_entropy = np.log2(K)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(range(T), slot_entropies, color="steelblue", edgecolor="white")
ax.axhline(max_entropy, color="red", linestyle="--", label=f"Max entropy ({max_entropy:.1f} bits)")
ax.set_xlabel("Token slot")
ax.set_ylabel("Entropy (bits)")
ax.set_title("Per-Slot Codebook Usage Entropy")
ax.set_xticks(range(T))
ax.legend()

for i, ent in enumerate(slot_entropies):
    n_effective = 2 ** ent
    ax.text(i, ent + 0.1, f"{n_effective:.0f}", ha="center", fontsize=9, color="gray")
ax.set_ylim(0, max_entropy + 1)
ax.text(T - 0.5, max_entropy + 0.5, "numbers = effective codebook size (2^H)", fontsize=8, ha="right", color="gray")

plt.tight_layout()
plt.show()

for slot, ent in enumerate(slot_entropies):
    n_used = (usage_matrix[slot] > 0).sum()
    print(f"Slot {slot}: H={ent:.2f} bits, effective={2**ent:.0f} codes, actually used={n_used}/{K}")

## 3. Slot Overlap (Jaccard similarity)

Do different slots use the same codes or different ones?
High Jaccard = slots share codes (less specialization). Low = distinct vocabularies per slot.

In [ ]:
# Jaccard similarity between slot vocabularies
slot_sets = [set(np.where(usage_matrix[s] > 0)[0]) for s in range(T)]
jaccard = np.zeros((T, T))
for i in range(T):
    for j in range(T):
        inter = len(slot_sets[i] & slot_sets[j])
        union = len(slot_sets[i] | slot_sets[j])
        jaccard[i, j] = inter / union if union > 0 else 0

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(jaccard, cmap="YlOrRd", vmin=0, vmax=1)
for i in range(T):
    for j in range(T):
        ax.text(j, i, f"{jaccard[i,j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_xticks(range(T))
ax.set_yticks(range(T))
ax.set_xlabel("Slot")
ax.set_ylabel("Slot")
ax.set_title("Slot Vocabulary Overlap (Jaccard)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 4. Code Co-occurrence Matrix (K x K)

How often do two codebook entries appear together in the same 8-token frame?
Strong off-diagonal pairs = compositional structure (e.g., code A = "left side" always with code B = "right side").

In [ ]:
# Co-occurrence: for each frame, count all pairs of codes present
cooccurrence = np.zeros((K, K), dtype=np.int64)
for frame_tokens in all_tokens.numpy():  # (T,)
    unique_codes = np.unique(frame_tokens)
    for i, ci in enumerate(unique_codes):
        for cj in unique_codes[i:]:
            cooccurrence[ci, cj] += 1
            if ci != cj:
                cooccurrence[cj, ci] += 1

# Only show the codes that are actually used (skip dead ones)
used_codes = np.where(cooccurrence.diagonal() > 0)[0]
cooc_sub = cooccurrence[np.ix_(used_codes, used_codes)]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(np.log1p(cooc_sub), aspect="auto", cmap="magma", interpolation="nearest")
ax.set_xlabel(f"Codebook index (active codes, n={len(used_codes)})")
ax.set_ylabel(f"Codebook index (active codes)")
ax.set_title("Code Co-occurrence (log scale)")
plt.colorbar(im, ax=ax, label="log(1 + count)")

# Tick every 10th used code for readability
tick_step = max(1, len(used_codes) // 20)
ax.set_xticks(range(0, len(used_codes), tick_step))
ax.set_xticklabels(used_codes[::tick_step], fontsize=7, rotation=45)
ax.set_yticks(range(0, len(used_codes), tick_step))
ax.set_yticklabels(used_codes[::tick_step], fontsize=7)

plt.tight_layout()
plt.show()

print(f"Active codes: {len(used_codes)}/{K} ({len(used_codes)/K*100:.1f}%)")
print(f"Dead codes: {K - len(used_codes)}")

## 5. Token-to-Pitch Mapping

For each codebook entry, average the ground-truth player positions of all frames that use it.
This reveals what each code "represents" spatially — e.g., a code might correspond to "compact defense on the left".

In [ ]:
def draw_pitch(ax):
    """Draw a soccer pitch outline."""
    ax.set_xlim(-2, PITCH_X + 2)
    ax.set_ylim(-2, PITCH_Y + 2)
    ax.set_aspect("equal")
    ax.set_facecolor("#2e8b57")
    pitch = plt.Rectangle((0, 0), PITCH_X, PITCH_Y, fill=False, edgecolor="white", linewidth=1.5)
    ax.add_patch(pitch)
    ax.plot([PITCH_X/2, PITCH_X/2], [0, PITCH_Y], color="white", linewidth=1)
    center_circle = plt.Circle((PITCH_X/2, PITCH_Y/2), 9.15, fill=False, edgecolor="white", linewidth=1)
    ax.add_patch(center_circle)
    for x_start in [0, PITCH_X - 16.5]:
        pa = plt.Rectangle((x_start, (PITCH_Y - 40.3)/2), 16.5, 40.3, fill=False, edgecolor="white", linewidth=1)
        ax.add_patch(pa)
    ax.set_xticks([])
    ax.set_yticks([])

def denormalize(x_norm, y_norm):
    x = x_norm * (PITCH_X / 2) + (PITCH_X / 2)
    y = y_norm * (PITCH_Y / 2) + (PITCH_Y / 2)
    return x, y

In [ ]:
# For the top-N most used codes overall, show the average pitch layout
# We pick one specific slot to keep it interpretable

SLOT_TO_SHOW = 0  # change this to explore other slots
TOP_N_CODES = 8   # how many codes to visualize

slot_codes = all_tokens[:, SLOT_TO_SHOW].numpy()
code_counts = Counter(slot_codes)
top_codes = [c for c, _ in code_counts.most_common(TOP_N_CODES)]

n_cols = 4
n_rows = (TOP_N_CODES + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.5 * n_rows))
axes = axes.flatten()
fig.patch.set_facecolor("#1a1a2e")

for idx, code in enumerate(top_codes):
    ax = axes[idx]
    draw_pitch(ax)

    # Find all frames where this slot used this code
    frame_mask = slot_codes == code
    gt_subset = all_gt[frame_mask]  # (N_matches, N_max, 4)

    # Average positions (first 2 features = x_norm, y_norm)
    avg_feats = gt_subset.mean(dim=0)  # (N_max, 4)
    n_players = min(22, avg_feats.shape[0])

    for p in range(n_players):
        x_n, y_n, team, ball = avg_feats[p].tolist()
        x, y = denormalize(x_n, y_n)
        color = "#e74c3c" if team > 0.5 else "#3498db"
        ax.scatter(x, y, c=color, s=60, edgecolors="white", linewidths=0.8, zorder=5, alpha=0.9)

    count = code_counts[code]
    ax.set_title(f"Slot {SLOT_TO_SHOW}, Code {code} (n={count})", fontsize=10, color="white", fontweight="bold")

# Hide unused subplots
for idx in range(TOP_N_CODES, len(axes)):
    axes[idx].set_visible(False)

fig.suptitle(f"Average pitch layout per code (slot {SLOT_TO_SHOW})", fontsize=14, color="white", fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## 6. Individual frames for a code (not just average)

The average can hide variance. Let's also show a few individual frames for a specific code to see how consistent the pattern is.

In [ ]:
# Show N_EXAMPLES individual frames for a specific (slot, code) pair
SLOT_INSPECT = 0
CODE_INSPECT = top_codes[0]  # most common code for SLOT_TO_SHOW
N_EXAMPLES = 6

frame_mask = all_tokens[:, SLOT_INSPECT].numpy() == CODE_INSPECT
matching_indices = np.where(frame_mask)[0]
rng = np.random.default_rng(42)
sample_idx = rng.choice(matching_indices, size=min(N_EXAMPLES, len(matching_indices)), replace=False)

fig, axes = plt.subplots(1, len(sample_idx), figsize=(5 * len(sample_idx), 4.5))
if len(sample_idx) == 1:
    axes = [axes]
fig.patch.set_facecolor("#1a1a2e")

for col, fi in enumerate(sample_idx):
    ax = axes[col]
    draw_pitch(ax)
    feats = all_gt[fi]  # (N_max, 4)
    n_players = min(22, feats.shape[0])
    for p in range(n_players):
        x_n, y_n, team, ball = feats[p].tolist()
        x, y = denormalize(x_n, y_n)
        color = "#e74c3c" if team > 0.5 else "#3498db"
        edge = "yellow" if ball > 0.5 else "white"
        ax.scatter(x, y, c=color, s=60, edgecolors=edge, linewidths=1.2, zorder=5)
    # Show all 8 tokens for this frame
    frame_codes = all_tokens[fi].tolist()
    ax.set_title(f"Frame {fi}\ntokens={frame_codes}", fontsize=8, color="white")

fig.suptitle(f"Individual frames with slot {SLOT_INSPECT} = code {CODE_INSPECT}", fontsize=13, color="white", fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

## 7. Codebook embedding space (t-SNE)

Project the K codebook vectors to 2D. Color by which slot uses them most — reveals if the codebook is partitioned per slot or shared.

In [ ]:
from sklearn.manifold import TSNE

codebook_weights = model.quantizer.codebook.weight.detach().cpu().numpy()  # (K, D)

# Dominant slot for each code: which slot uses it most?
dominant_slot = np.argmax(usage_matrix, axis=0)  # (K,)
total_usage = usage_matrix.sum(axis=0)            # (K,)
is_alive = total_usage > 0

# t-SNE on alive codes only
alive_idx = np.where(is_alive)[0]
tsne = TSNE(n_components=2, perplexity=min(30, len(alive_idx) - 1), random_state=42)
coords = tsne.fit_transform(codebook_weights[alive_idx])

fig, ax = plt.subplots(figsize=(10, 8))
cmap = plt.cm.tab10
for slot in range(T):
    mask = dominant_slot[alive_idx] == slot
    if mask.any():
        sizes = np.log1p(total_usage[alive_idx][mask]) * 10
        ax.scatter(coords[mask, 0], coords[mask, 1], c=[cmap(slot)],
                   s=sizes, label=f"Slot {slot}", alpha=0.7, edgecolors="white", linewidths=0.3)

ax.legend(title="Dominant slot", fontsize=9)
ax.set_title("Codebook t-SNE (colored by dominant slot, sized by usage)")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
plt.tight_layout()
plt.show()

## 8. Summary statistics

In [ ]:
print("=" * 60)
print("VQ-VAE INTERPRETABILITY SUMMARY")
print("=" * 60)

n_frames = all_tokens.shape[0]
n_alive = int(is_alive.sum())
n_dead = K - n_alive

# Unique 8-token sequences
unique_sequences = len(set(tuple(row) for row in all_tokens.numpy()))

# Average within-frame token diversity (how many distinct codes per frame)
per_frame_unique = np.array([len(set(row)) for row in all_tokens.numpy()])

print(f"\nDataset: {n_frames} frames, {SPLIT} split")
print(f"Codebook: {n_alive}/{K} codes alive ({n_dead} dead)")
print(f"Unique 8-token sequences: {unique_sequences} / {n_frames} frames ({unique_sequences/n_frames*100:.1f}%)")
print(f"Per-frame distinct tokens: mean={per_frame_unique.mean():.1f}, min={per_frame_unique.min()}, max={per_frame_unique.max()} (out of {T})")
print(f"\nPer-slot entropy (bits):")
for slot, ent in enumerate(slot_entropies):
    print(f"  Slot {slot}: {ent:.2f} / {max_entropy:.2f}  (effective codes: {2**ent:.0f})")
print(f"\nMean slot entropy: {np.mean(slot_entropies):.2f} bits")
print(f"Total information capacity: {T} slots x {np.mean(slot_entropies):.1f} bits/slot = {T * np.mean(slot_entropies):.1f} bits/frame")